In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dataset-chatbot-2/merge_output.json


In [2]:
!pip install bitsandbytes
!pip install -U datasets huggingface_hub fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 26.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found

In [3]:
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    HfArgumentParser, TrainingArguments, pipeline, logging, Trainer
)

from peft import LoraConfig, get_peft_model, TaskType, PeftModel

2025-06-26 06:35:24.911045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750919725.100929      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750919725.159889      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
model_name = 'vilm/vinallama-2.7b-chat'

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = 'auto',
    trust_remote_code = True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(base_model)

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(46306, 2560, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2560, out_features=2560, bias=False)
          (k_proj): Linear4bit(in_features=2560, out_features=2560, bias=False)
          (v_proj): Linear4bit(in_features=2560, out_features=2560, bias=False)
          (o_proj): Linear4bit(in_features=2560, out_features=2560, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=6912, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=6912, bias=False)
          (down_proj): Linear4bit(in_features=6912, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2560,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2560,), eps=1e-05)
      )
    )
    (norm): LlamaR

In [5]:
lora_config = LoraConfig(
    r = 64,
    lora_alpha = 16,
    lora_dropout = 0.1,
    bias = 'none',
    target_modules =[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type = TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)

In [6]:
# data = load_dataset('openai/gsm8k', 'main', split='train[:200]')
# Load dữ liệu jsonl
data = load_dataset('json', data_files='/kaggle/input/dataset-chatbot-2/merge_output.json')
dataset = data['train']
print(dataset.column_names)
# Tách train/test
# split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
# train_dataset = split_dataset['train']
# test_dataset = split_dataset['test']

Generating train split: 0 examples [00:00, ? examples/s]

['Question', 'Answer']


In [7]:
print(dataset[0])

{'Question': 'Làm thế nào để chọn kem nền phù hợp cho da mụn?', 'Answer': 'Để chọn kem nền cho da mụn cần ưu tiên sản phẩm không gây bít tắc lỗ chân lông (non-comedogenic) có công thức nhẹ kiểm soát dầu và chứa thành phần hỗ trợ trị mụn như niacinamide hoặc axit salicylic. Xác định undertone da (ấm lạnh trung tính) bằng cách kiểm tra tĩnh mạch cổ tay: xanh lá là ấm xanh dương là lạnh cả hai là trung tính. Thử kem nền trên vùng xương hàm dưới ánh sáng tự nhiên chọn màu hòa quyện với da cổ không để lại ranh giới. Tránh kem nền quá dày hoặc dầu vì dễ kích ứng mụn. Chọn finish bán lì (satin) hoặc lì (matte) để che phủ tốt nhưng không bóng nhờn. Thoa thử và quan sát sau 4-6 giờ để kiểm tra độ bền có bị trôi hoặc kích ứng không. Dùng mút ẩm tán kem mỏng tập trung vào vùng đều màu tránh chà xát vùng mụn. Kết hợp kem lót kiềm dầu và phấn phủ không chứa talc để giữ lớp nền mịn giảm nguy cơ bùng phát mụn.'}


In [10]:
def tokenize(batch):
    texts = [
        f'<s>[INST] {user_text} [/INST] {ai_text} </s>'
        for user_text, ai_text in zip (batch["Question"], batch["Answer"])
    ]
    tokens = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=400,
        return_tensors='pt'
    )

    tokens['labels'] = tokens['input_ids'].clone()

    return tokens

In [11]:
tokenized_data = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names, batch_size=8)

Map:   0%|          | 0/8439 [00:00<?, ? examples/s]

In [12]:
print(tokenized_data[0])

{'input_ids': [1, 1, 518, 25580, 29962, 34282, 32249, 32389, 32073, 32742, 35287, 33106, 32969, 32180, 3060, 1146, 36099, 29973, 518, 29914, 25580, 29962, 33202, 32742, 35287, 33106, 3060, 1146, 36099, 32420, 33261, 32569, 32214, 32489, 32035, 32518, 40773, 33460, 33993, 32795, 34417, 313, 5464, 29899, 510, 287, 6352, 293, 29897, 28810, 32066, 32449, 33071, 32453, 33001, 33157, 32006, 33440, 32140, 32361, 32727, 32559, 32447, 36099, 32063, 302, 13544, 40687, 680, 32490, 36920, 4497, 4245, 506, 29889, 35352, 32087, 22332, 650, 1146, 313, 32611, 33533, 32498, 32384, 29897, 32373, 32338, 32453, 1020, 34435, 34049, 32533, 32462, 29901, 33178, 24303, 18916, 33944, 33178, 33820, 18916, 33533, 32124, 32288, 18916, 32498, 32384, 29889, 37191, 35287, 33106, 260, 42944, 29876, 32684, 34226, 34489, 32807, 33308, 32558, 32336, 32377, 32742, 32730, 32979, 39355, 32034, 1146, 32533, 32035, 32073, 32125, 35087, 32368, 29889, 38963, 35287, 33106, 32430, 34174, 32490, 33157, 32317, 33005, 33232, 32568,

In [13]:
training_args = TrainingArguments(
    output_dir = './lora-tuned',
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    optim = "paged_adamw_32bit",
    learning_rate = 2e-5,
    num_train_epochs = 5,
    fp16 = False,
    bf16 = False,
    max_grad_norm = 0.3,
    lr_scheduler_type = "cosine", 
    max_steps = -1, 
    warmup_ratio = 0.03,
    weight_decay=0.01,
    logging_steps = 20,
    save_strategy = 'epoch',
    report_to = 'none',
    remove_unused_columns = False,      
    label_names = ["labels"]
)

In [14]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_data,
    processing_class = tokenizer
)

In [ ]:
trainer.train()

Step,Training Loss
20,10.363700
40,8.591200
60,2.248000
80,0.690300
100,0.621200
120,0.552200
140,0.488400
160,0.445300
180,0.415100
200,0.379900


In [ ]:
finetune_model = "Llama-2-7b-chat-finetune"
trainer.model.save_pretrained(finetune_model)

In [ ]:
from tensorboard import notebook
log_dir = "lora-tuned/runs"
notebook.start("--logdir {} --port 4000".format(log_dir))

In [ ]:
logging.set_verbosity(logging.CRITICAL)

prompt = "Trang điểm như thế nào để đẹp như diễn viên?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=250)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

In [ ]:
import gc
del model, pipe, trainer

gc.collect()

In [ ]:
# Reload and merge
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map = 'auto'
)
model = PeftModel.from_pretrained(base_model, finetune_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
model.save_pretrained("models/finetune_model.pt")
tokenizer.save_pretrained("models/tokenizer/")

In [ ]:
!zip -r my_model.zip /kaggle/working/models/finetune_model.pt/model-00001-of-00002.safetensors
